*0.3 Classical NLP*

# Tokenization: word

**The situation.** A support analytics job counts words per ticket to flag unusually long ones and to build a word cloud of top complaints. It uses `text.split()`. The word cloud's top entry is `"charged."` — with the full stop — separate from `"charged"`; `"don't"` appears as one token; and `"$1,250.50"` is one "word". The counts are wrong in ways nobody notices for months.

**Word tokenization.** Splitting text into words is the first step of every classical NLP pipeline, and it is harder than splitting on spaces. Punctuation, contractions, money, abbreviations all need rules. A real tokenizer — NLTK's or spaCy's — has those rules built in.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
from nltk.tokenize import word_tokenize

ticket = "Don't charge me $1,250.50 twice, U.S. customers said."
naive = ticket.split()
proper = word_tokenize(ticket)
print("split():       ", naive)
print("word_tokenize():", proper)
assert "charge" in proper and "said" in proper and "said." not in proper

split():        ["Don't", 'charge', 'me', '$1,250.50', 'twice,', 'U.S.', 'customers', 'said.']
word_tokenize(): ['Do', "n't", 'charge', 'me', '$', '1,250.50', 'twice', ',', 'U.S.', 'customers', 'said', '.']


**Reading the output.** `split()` glues punctuation to words (`"twice,"`, `"said."`) and leaves the contraction whole. `word_tokenize` separates the comma and full stop, splits `"Don't"` into `"Do"` + `"n't"` (two grammatical pieces), and keeps `"$1,250.50"` and `"U.S."` intact because its rules know about money and abbreviations.

**The word cloud, fixed.** Count with the proper tokens; punctuation and case no longer split the counts.

In [3]:
from collections import Counter

tickets = [
    "Charged twice for my order.",
    "I was charged twice!",
    "Why am I charged twice, again?",
    "Refund for the duplicate charge please.",
]
naive_counts = Counter()
proper_counts = Counter()
for text in tickets:
    naive_counts.update(text.lower().split())
    for token in word_tokenize(text.lower()):
        if token.isalpha():  # drop punctuation tokens
            proper_counts[token] += 1
print(
    "split():        charged =",
    naive_counts["charged"],
    "| 'charged.' =",
    naive_counts["charged."],
    "| 'twice!' =",
    naive_counts["twice!"],
)
print("word_tokenize(): charged =", proper_counts["charged"], "| twice =", proper_counts["twice"])
assert proper_counts["charged"] == 3 and proper_counts["twice"] == 3

split():        charged = 3 | 'charged.' = 0 | 'twice!' = 1
word_tokenize(): charged = 3 | twice = 3


**Reading the output.** With `split()`, "charged" is spread across several spellings; with a tokenizer, all three count together.

**The rule to remember.** Never `split()` for real text. Use a tokenizer with rules — NLTK's `word_tokenize` or spaCy — and decide explicitly what to do with punctuation and case.

| Use it when | Don't when | Instead use |
|---|---|---|
| counting, keyword search, classical models (TF-IDF, BM25) | feeding a neural model — they use subword tokens (next items) | the model's own tokenizer |

**Watch out**
- Tokenizers are language-specific. English rules on German or Chinese text produce nonsense.
- `"n't"` as a token surprises people; it is deliberate (so "do" and "not" are analysable separately). Normalise if you count negations.
- Lowercasing merges "US" and "us". Decide per use case; do not lowercase before NER.